# 🔮 Reto Completo: Del Dataset al Despliegue de Machine Learning (Ruta B)
### Google Colab + Streamlit Hyper-Visual + GitHub + Docker + Google Cloud Compute Engine

**Ruta B:** Aprendizaje No Supervisado - Segmentación de Clientes con K-Means ($k=3$).
Este notebook permite ejecutar todo el flujo desde Google Colab de manera 100% reproducible.

## 1. Instalación de Dependencias en Google Colab

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy joblib plotly matplotlib openpyxl

## 2. Creación de la Estructura de Proyecto

In [ ]:
from pathlib import Path

PROJECT = Path('/content/ml_clustering_project')
for sub in ['datasets', 'data', 'graphs', 'src', 'docs']:
    (PROJECT / sub).mkdir(parents=True, exist_ok=True)

print('✔ Estructura creada en:', PROJECT)

## 3. Carga y Exploración del Dataset

In [ ]:
import pandas as pd

# Descargar o cargar el dataset
DATA_URL = 'https://raw.githubusercontent.com/datasets/telecom-churn/master/customer-churn.csv'
# Alternativa: leer dataset local si ya fue subido
df = pd.read_csv('datasets/datos.csv') if Path('datasets/datos.csv').exists() else pd.read_csv(DATA_URL)
df.columns = [c.strip().replace('  ', ' ') for c in df.columns]

# Guardar copia local en el proyecto
df.to_csv(PROJECT / 'datasets/datos.csv', index=False)
df.to_csv(PROJECT / 'data/datos.csv', index=False)

print('Dimensiones:', df.shape)
display(df.head())
display(df.describe().T)

## 4. Pipeline Hermético y Entrenamiento (K-Means k=3)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score

features = [
    'Subscription Length',
    'Charge Amount',
    'Seconds of Use',
    'Frequency of use',
    'Frequency of SMS',
    'Distinct Called Numbers',
    'Age',
    'Customer Value'
]
X = df[features].copy()

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('cluster', KMeans(n_clusters=3, random_state=42, n_init=10))
])

labels = model.fit_predict(X)
X_scaled = model[:-1].transform(X)

sil = silhouette_score(X_scaled, labels)
ch = calinski_harabasz_score(X_scaled, labels)
print(f'Silhouette Score: {sil:.4f}')
print(f'Calinski-Harabasz: {ch:.2f}')

## 5. Interpretación de Perfiles de Clusters

In [ ]:
df_clustered = df.copy()
df_clustered['cluster'] = labels
profiles = df_clustered.groupby('cluster')[features].mean().round(2)
display(profiles)

## 6. Ejecución de `train.py` para Serializar `model.joblib`

In [ ]:
%cd /content/ml_clustering_project
!python train.py

## 7. Probar Streamlit en Colab usando Túnel Temporal (Localtunnel)

In [ ]:
# Iniciar Streamlit en segundo plano
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &

# Verificar túnel con localtunnel
!npm install -g localtunnel
!npx localtunnel --port 8501

## 8. Empaquetar y Descargar para Despliegue en GCP

In [ ]:
!cd /content && zip -r ml_clustering_project.zip ml_clustering_project
from google.colab import files
files.download('/content/ml_clustering_project.zip')